In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkFiles

# Inicializar la sesión de Spark
spark = SparkSession.builder.appName("ClasificadorTitanic").getOrCreate()

# Utilizamos la URL 'raw' de GitHub para descargar el archivo directamente
url_titanic = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/main/titanic.csv"
spark.sparkContext.addFile(url_titanic)

# Leer el CSV
df_titanic = spark.read.csv(SparkFiles.get("titanic.csv"), header=True, inferSchema=True)

# Seleccionar predictores clave y eliminar valores nulos para simplificar el entrenamiento
df_modelo = df_titanic.select("Survived", "Pclass", "Sex", "Age", "Fare").dropna()
df_modelo.show(5)

+--------+------+------+----+-------+
|Survived|Pclass|   Sex| Age|   Fare|
+--------+------+------+----+-------+
|       0|     3|  male|22.0|   7.25|
|       1|     1|female|38.0|71.2833|
|       1|     3|female|26.0|  7.925|
|       1|     1|female|35.0|   53.1|
|       0|     3|  male|35.0|   8.05|
+--------+------+------+----+-------+
only showing top 5 rows


Preprocesamiento de los datos

In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Convertir la columna de texto 'Sex' (male/female) a índice numérico (0.0 / 1.0)
indexer = StringIndexer(inputCol="Sex", outputCol="Sex_index")
df_indexado = indexer.fit(df_modelo).transform(df_modelo)

# 2. Agrupar las variables independientes en una única columna llamada 'features'
assembler = VectorAssembler(
    inputCols=["Pclass", "Sex_index", "Age", "Fare"],
    outputCol="features"
)
df_final = assembler.transform(df_indexado)

# 3. Dividir el dataset: 80% para entrenar el modelo y 20% para evaluarlo
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

Entrenamiento y evaluación del modelo Random Forest

In [3]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Inicializar el clasificador con 50 árboles de decisión
rf = RandomForestClassifier(featuresCol="features", labelCol="Survived", numTrees=50, maxDepth=5, seed=42)

# 2. Entrenar el modelo con los datos de entrenamiento
modelo_rf = rf.fit(train_data)

# 3. Realizar predicciones sobre los datos de prueba
predicciones = modelo_rf.transform(test_data)

# Mostrar resultados: 'probability' muestra la certeza del modelo (ej. 70% vivo, 30% fallecido)
predicciones.select("Pclass", "Sex_index", "Age", "Survived", "prediction", "probability").show(5)

# 4. Calcular la precisión global del modelo
evaluator = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="accuracy")
precision = evaluator.evaluate(predicciones)

print(f"Precisión (Accuracy) del Random Forest: {precision * 100:.2f}%")

+------+---------+----+--------+----------+--------------------+
|Pclass|Sex_index| Age|Survived|prediction|         probability|
+------+---------+----+--------+----------+--------------------+
|     1|      1.0|50.0|       0|       1.0|[0.07916054111086...|
|     1|      0.0|21.0|       0|       0.0|[0.61865390737211...|
|     1|      0.0|24.0|       0|       0.0|[0.61530804271046...|
|     1|      0.0|29.0|       0|       0.0|[0.65741880478093...|
|     1|      0.0|36.0|       0|       0.0|[0.66004166281621...|
+------+---------+----+--------+----------+--------------------+
only showing top 5 rows
Precisión (Accuracy) del Random Forest: 83.33%


### Entendiendo las Métricas de Clasificación y la Matriz de Confusión

Después de entrenar y evaluar el modelo, es crucial entender cómo de bien funciona sobre nuevos datos distintos a los usados para el entrenamiento. Además de la precisión global ya calculado, existen otras métricas para obtener una visión más completa, especialmente en problemas donde las clases pueden estar desequilibradas (como el de predecir la supervivencia en el Titanic).

Veamos algunas otras métricas:

1.  **Métricas Adicionales de Clasificación (F1-Score, Precisión Ponderada, Recall Ponderado):**
    *   **F1-Score:** Es una media armónica de la precisión y el recall. Es útil cuando se busca un equilibrio entre ambas métricas y para datasets desequilibrados.
    *   **Precisión Ponderada (Weighted Precision):** Mide la proporción de verdaderos positivos entre los positivos predichos, pero considerando el soporte (número de instancias) de cada clase. Es útil para tener una visión general de la precisión en todas las clases.
    *   **Recall Ponderado (Weighted Recall):** Mide la proporción de verdaderos positivos entre todos los positivos reales, también ponderado por el soporte de cada clase. Es útil para ver cómo de bien el modelo identifica todas las instancias positivas.

2.  **Matriz de Confusión:**
    *   Es una tabla que permite visualizar el rendimiento de un algoritmo de clasificación. Cada fila representa las instancias en una clase predicha, mientras que cada columna representa las instancias en una clase real (o viceversa).
    *   Para el problema del Titanic:
        *   `Actual 0`: Personas que realmente no sobrevivieron.
        *   `Actual 1`: Personas que realmente sobrevivieron.
        *   `Predicted 0`: Predicciones de que la persona no sobrevivió.
        *   `Predicted 1`: Predicciones de que la persona sobrevivió.
    *   Nos ayuda a identificar:
        *   **Verdaderos Positivos (TP):** Predicho 1, Actual 1 (¡Sobrevivió y lo predijimos correctamente!).
        *   **Verdaderos Negativos (TN):** Predicho 0, Actual 0 (No sobrevivió y lo predijimos correctamente).
        *   **Falsos Positivos (FP):** Predicho 1, Actual 0 (Error: Predijimos que sobrevivió, pero no).
        *   **Falsos Negativos (FN):** Predicho 0, Actual 1 (Error: Predijimos que no sobrevivió, pero sí).

Para calcular la matriz de confusión con PySpark, a menudo necesitamos convertir las predicciones y las etiquetas a un formato RDD (`predictionAndLabels`) para usar las herramientas de `pyspark.mllib.evaluation`, que están optimizadas para este tipo de cálculo.

In [5]:
from pyspark.mllib.evaluation import MulticlassMetrics
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Inicializar el evaluador una sola vez
evaluator = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction")

# Calcular métricas adicionales
f1_score = evaluator.evaluate(predicciones, {evaluator.metricName: "f1"})
weighted_precision = evaluator.evaluate(predicciones, {evaluator.metricName: "weightedPrecision"})
weighted_recall = evaluator.evaluate(predicciones, {evaluator.metricName: "weightedRecall"})

print(f"F1-Score: {f1_score * 100:.2f}%")
print(f"Precisión Ponderada: {weighted_precision * 100:.2f}%")
print(f"Recall Ponderado: {weighted_recall * 100:.2f}%\n")

# Calcular la Matriz de Confusión
# Para la matriz de confusión, necesitamos un RDD de (prediction, label) para MulticlassMetrics
predictionAndLabels = predicciones.select("prediction", "Survived").rdd.map(lambda row: (float(row.prediction), float(row.Survived)))

# Instanciar el objeto de métricas
metrics = MulticlassMetrics(predictionAndLabels)

# Obtener la Matriz de Confusión
confusion_matrix_rdd = metrics.confusionMatrix()

# Convertir a Pandas DataFrame para una mejor visualización en Colab
confusion_matrix_pd = pd.DataFrame(
    confusion_matrix_rdd.toArray(),
    index=['Actual 0 (No Sobrevive)', 'Actual 1 (Sobrevive)'],
    columns=['Predicho 0 (No Sobrevive)', 'Predicho 1 (Sobrevive)']
)

print("Matriz de Confusión:")
display(confusion_matrix_pd)

F1-Score: 83.09%
Precisión Ponderada: 84.47%
Recall Ponderado: 83.33%



/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Matriz de Confusión:


,Predicho 0 (No Sobrevive),Predicho 1 (Sobrevive)
Actual 0 (No Sobrevive),56.0,4.0
Actual 1 (Sobrevive),15.0,39.0
